# NOVA Smart Tourism Platform: Travel Logistics & Availability Agent
### Agentic AI Component - Production Implementation & Architecture

---

## 1. Project Context & Multi-Agent Architecture

The **NOVA Smart Tourism Platform** is an intelligent travel assistance platform specifically designed for tourism in Sri Lanka. The multi-agent architecture decomposes complex travel planning into four specialized, autonomous, and collaborating agents:

1. **Travel Planning Agent**: Coordinates overall travel itineraries, manages daily timeline scheduling, and integrates outputs from other specialized agents.
2. **Destination Research Agent**: Researches factual destination context, attractions, activities, and entrance fees.
3. **Recommendation & Feedback Analysis Agent**: Analyzes tourist reviews and sentiment to deliver personalized attraction recommendations and suitability scoring.
4. **Travel Logistics & Availability Agent** *(THIS IMPLEMENTATION)*: Obtains and structures factual transport routes, public transit options (bus numbers, train routes), timetables, travel durations, transit feasibility, and availability guidance.

---

### Core Architectural Capabilities:
- **LLM Tool Calling & Structured Schemas**: Structured input/output validation for deterministic logistics retrieval.
- **RAG & Vector Search**: FAISS vector indexing with MiniLM dense embeddings for transport schedules, bus routes, and transit times.
- **Stateful Agent Workflow**: LangGraph state graph with bounded execution (`MAX_STEPS`), safe operational tracing, and robust error recovery.

---

### System Architecture Diagram

```mermaid
graph TD
    User([User / Travel Planning Agent]) --> Agent[Travel Logistics & Availability Agent]
    
    subgraph Guardrails & LangGraph State Workflow
        Agent --> Decision{Requires Tool Call?}
        Decision -- Yes --> ToolExecution[Execute Selected Tool]
        ToolExecution --> Agent
        Decision -- No / Bounded Limit --> Validation[Output Validation Node]
        Validation --> FinalOutput[Validated LogisticsResult JSON]
    end

    subgraph Tool Layer & Service Abstractions
        ToolExecution --> SearchLogistics[search_logistics_information]
        ToolExecution --> SearchRoutes[search_transport_routes]
        ToolExecution --> SearchSchedules[search_transport_schedule]
        ToolExecution --> CheckAvail[check_transport_availability]
        ToolExecution --> SearchHours[search_attraction_hours]
    end

    subgraph RAG & Knowledge Base
        SearchLogistics --> RAGRetriever[FAISS Vector Store]
        RAGRetriever --> MiniLM[HuggingFace sentence-transformers/all-MiniLM-L6-v2]
        RAGRetriever --> KnowledgeBase[(Prototype Logistics Knowledge Base)]
    end

    subgraph Future Extensibility - Clean Service Layer
        SearchRoutes -.-> RouteService[RouteService / Google Maps API]
        SearchSchedules -.-> TransportService[TransportService / SL Railways & NTC Bus APIs]
        CheckAvail -.-> AvailabilityService[AvailabilityService / Live Reservation Engine]
    end
```

---
## Section 2: Agent Responsibilities & Boundary Guardrails

In multi-agent systems, strict adherence to the **Single Responsibility Principle** prevents hallucinations, state conflicts, and redundant computations.

| Capability | In-Scope for Travel Logistics Agent (✅) | Out-of-Scope (❌ Handled by Other Agents) |
| :--- | :--- | :--- |
| **Transport Research** | Finding trains, buses, and highway options between cities | Planning multi-day travel itineraries (Travel Planning Agent) |
| **Public Transit** | Identifying official bus routes/numbers and train names | Deciding which destination is best for the user (Recommendation Agent) |
| **Transit Feasibility** | Timetables, durations, distances, and attraction open hours | Review sentiment analysis and tourist scoring (Feedback Agent) |
| **Availability** | Reporting scheduled service availability patterns | Performing live ticket booking or financial transactions |
| **Knowledge Boundary**| Clearly declaring unverified live seat availability | Fabricating real-time seat counts or missing routes |

> [!IMPORTANT]
> **Safety Guardrail**: This agent will refuse to generate full itineraries, will refuse to score tourist destination suitability, and will clarify that booking functionality is not connected.

---
## Section 3: Environment Setup & Compatibility Verification

### What we are building:
We verify Python version and test system runtime environment.

### Why we need it:
Agentic AI components require exact package interoperability (LangChain 1.4+, LangGraph, Pydantic v2).

In [1]:
# Section 3: Environment Setup
import sys
import platform

print(f"🐍 Python Version: {platform.python_version()} on {platform.system()} ({platform.machine()})")
assert sys.version_info >= (3, 10), "Python 3.10+ is required for modern typing and LangChain 1.x support."
print("✅ Runtime environment verified.")

🐍 Python Version: 3.14.6 on Windows (AMD64)
✅ Runtime environment verified.


---
## Section 4: Imports

### What we are building:
Import all required modules from LangChain 1.x, LangGraph, Pydantic, and Google GenAI.

### Why we need it:
- `langchain_google_genai`: Integration for modern Gemini LLMs (`gemini-2.5-flash-lite`).
- `langchain_huggingface` & `sentence_transformers`: Dense vector representations.
- `langchain_community.vectorstores.FAISS`: Efficient local in-memory similarity search.
- `langgraph`: Declarative stateful agent execution graph with conditional edges.
- `pydantic`: Strong schema validation for inputs and structured JSON outputs.

In [2]:
# Section 4: Imports
import os
import sys
import json
import uuid
import types
from typing import Annotated, List, Optional, Sequence, TypedDict, Dict, Any

# Windows DLL Application Control policy compatibility fallback
try:
    import uuid_utils
except ImportError:
    u = types.ModuleType("uuid_utils")
    u.UUID = uuid.UUID
    uc = types.ModuleType("uuid_utils.compat")
    uc.uuid7 = uuid.uuid4
    sys.modules["uuid_utils"] = u
    sys.modules["uuid_utils.compat"] = uc

try:
    from sklearn.metrics.cluster import _expected_mutual_info_fast
except ImportError:
    m = types.ModuleType("sklearn.metrics.cluster._expected_mutual_info_fast")
    m.expected_mutual_information = lambda *a, **k: 0
    sys.modules["sklearn.metrics.cluster._expected_mutual_info_fast"] = m

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field, ValidationError

# LangChain Core elements
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage,
    ToolMessage
)

# LLM and Embeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangGraph for Agent Loop and State
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

print("✅ All modern LangChain and LangGraph dependencies loaded successfully!")

C:\Users\user\AppData\Local\Temp\ipykernel_12184\2436093715.py:44: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✅ All modern LangChain and LangGraph dependencies loaded successfully!


---
## Section 5: Environment Variables & Security

### What we are building:
Safely load `GOOGLE_API_KEY` from the project `.env` file without hardcoding or printing secrets.

### Why we need it:
Adheres to security best practices. Secrets must never be committed to Git or printed in notebook outputs.

In [3]:
# Section 5: Environment Variables & Security
# Locate .env either in current directory or parent directory
env_path = find_dotenv(filename=".env", usecwd=True)
if not env_path or not os.path.exists(env_path):
    # Fallback search in parent directory (ai-agents/.env)
    candidate = os.path.abspath(os.path.join("..", ".env"))
    if os.path.exists(candidate):
        env_path = candidate

if env_path and os.path.exists(env_path):
    load_dotenv(dotenv_path=env_path)
    print(f"✅ Loaded environment configuration from: {env_path}")
else:
    load_dotenv()
    print("⚠️ Standard load_dotenv() invoked.")

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("❌ GOOGLE_API_KEY is missing from environment. Please populate ai-agents/.env.")

print(f"🔒 Security Check Passed: GOOGLE_API_KEY is present (Length: {len(api_key)} chars, strictly hidden).")

✅ Loaded environment configuration from: c:\Users\user\NOVA\ai-agents\notebooks\.env
🔒 Security Check Passed: GOOGLE_API_KEY is present (Length: 53 chars, strictly hidden).


---
## Section 6: Gemini LLM Connection

### What we are building:
Instantiate the `ChatGoogleGenerativeAI` client using model `gemini-2.5-flash-lite` at `temperature=0`.

### Why we need it:
A temperature of `0` ensures deterministic, factual, and reproducible agent reasoning, preventing creative hallucinations when handling transport schedules and routes.

In [4]:
# Section 6: Gemini Connection
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

# Quick connectivity probe
probe_response = llm.invoke([HumanMessage(content="Respond with 'CONNECTED' if you receive this message.")])

# Safely extract text (langchain-google-genai returns a list of content blocks)
raw_probe = getattr(probe_response, "content", probe_response)
if isinstance(raw_probe, list):
    probe_text = " ".join(
        c.get("text", str(c)) if isinstance(c, dict) else str(c) for c in raw_probe
    )
else:
    probe_text = str(raw_probe)

print(f"🤖 LLM Probe Response: {probe_text.strip()}")
print(f"✅ Connected to Google Gemini (model: {llm.model}) at temperature=0.")


c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


🤖 LLM Probe Response: CONNECTED
✅ Connected to Google Gemini (model: gemini-3.5-flash-lite) at temperature=0.


---
## Section 7: Define Logistics Knowledge Base

### What we are building:
Define a structured prototype knowledge base of Sri Lankan travel corridors covering Colombo, Kandy, Ella, Galle, and Nuwara Eliya.

### Why we need it:
Provides verified ground-truth records for RAG retrieval. All records are explicitly attributed to `source = "prototype knowledge base"`.

In [5]:
# Section 7: Define Logistics Knowledge Base
PROTOTYPE_LOGISTICS_DATA = [
    {
        "id": "ROUTE_COL_KDY_TRAIN_01",
        "origin": "Colombo",
        "destination": "Kandy",
        "transport_type": "Train",
        "route": "Main Line (Colombo Fort - Badulla / Kandy)",
        "service_operator": "Sri Lanka Railways",
        "service_name": "Intercity & Express Trains (Podi Menike, Udarata Menike)",
        "bus_or_train_number": "Train 1005 / Train 1015 / Train 1009",
        "departure_location": "Colombo Fort Railway Station",
        "arrival_location": "Kandy Railway Station",
        "schedule_info": "Daily departures: 05:55 AM (Express), 07:00 AM (Intercity), 08:30 AM (Express), 10:35 AM (Podi Menike), 03:35 PM (Express), 05:45 PM (Intercity)",
        "estimated_duration": "2 hours 30 minutes to 3 hours 15 minutes",
        "estimated_distance_km": 115,
        "frequency": "6 to 8 daily scheduled trains",
        "cost_estimate": "1st Class Observation/AC: LKR 1,500 – 2,500; 2nd Class: LKR 500 – 800; 3rd Class: LKR 240",
        "availability_status": "Scheduled daily operations in prototype knowledge base. Real-time seat availability cannot be verified without live railway API integration.",
        "timing_feasibility_notes": "Morning trains departing before 08:30 AM allow arrival in Kandy before 12:00 PM, suitable for afternoon visits to Temple of the Tooth.",
        "notes": "Scenic ascent into central highlands. 1st and 2nd class reserved seats require advance booking at stations or via official reservation portals.",
        "source": "prototype knowledge base",
        "last_updated": "2026-03-01"
    },
    {
        "id": "ROUTE_COL_KDY_BUS_01",
        "origin": "Colombo",
        "destination": "Kandy",
        "transport_type": "Bus",
        "route": "Route 01 (Colombo - Kandy via A1 Highway)",
        "service_operator": "National Transport Commission (NTC) / SLTB / Private Operators",
        "service_name": "Route 01 Semi-Luxury / AC Intercity / Normal SLTB",
        "bus_or_train_number": "Bus Route 01",
        "departure_location": "Bastian Mawatha Bus Station (Pettah, Colombo)",
        "arrival_location": "Goodshed Bus Stand (Kandy)",
        "schedule_info": "Frequent daily departures every 15 to 30 minutes starting from 04:30 AM until 09:30 PM",
        "estimated_duration": "3 hours 15 minutes to 4 hours (subject to A1 traffic)",
        "estimated_distance_km": 116,
        "frequency": "Every 15–20 minutes during daytime",
        "cost_estimate": "AC Intercity: approx LKR 600 – 850; Normal SLTB: approx LKR 350",
        "availability_status": "Continuous walk-in availability on departure. Advance seat reservation not required for standard buses.",
        "timing_feasibility_notes": "Depart before 06:30 AM to avoid heavy suburban Colombo exit congestion.",
        "notes": "Fastest road option is the AC Intercity service departing from Bastian Mawatha.",
        "source": "prototype knowledge base",
        "last_updated": "2026-03-01"
    },
    {
        "id": "ROUTE_KDY_ELL_TRAIN_01",
        "origin": "Kandy",
        "destination": "Ella",
        "transport_type": "Train",
        "route": "Main Line Scenic Highland Railway (Kandy - Nanu Oya - Ella - Badulla)",
        "service_operator": "Sri Lanka Railways",
        "service_name": "Highland Scenic Train (Podi Menike / Udarata Menike / Ella Odyssey)",
        "bus_or_train_number": "Train 1005 (Podi Menike) / Train 1015 (Udarata Menike) / Train 1041 (Ella Odyssey)",
        "departure_location": "Kandy Railway Station / Peradeniya Junction Station",
        "arrival_location": "Ella Railway Station",
        "schedule_info": "Morning departure: approx 08:47 AM (from Peradeniya Junction) and 11:10 AM (from Kandy/Peradeniya); Ella Odyssey tourist train operates on designated weekly schedules (approx 07:00 AM from Colombo / 09:15 AM Peradeniya)",
        "estimated_duration": "6 hours to 6 hours 45 minutes",
        "estimated_distance_km": 140,
        "frequency": "3 to 4 scheduled daily services",
        "cost_estimate": "Reserved 1st Class: LKR 3,000 – 5,000; Reserved 2nd Class: LKR 1,500 – 2,000; Unreserved 3rd Class: LKR 350 – 500",
        "availability_status": "Scheduled daily operations in prototype knowledge base. Very high tourist demand; reserved seats frequently sell out 30 days prior. Real-time availability requires official railway portal verification.",
        "timing_feasibility_notes": "Full-day journey. Travelers arrive in Ella late afternoon (around 03:30 PM – 05:30 PM), meaning hikes like Little Adam's Peak are best scheduled for the next morning.",
        "notes": "Globally renowned scenic rail journey passing tea plantations, waterfalls, and mountain tunnels.",
        "source": "prototype knowledge base",
        "last_updated": "2026-03-01"
    },
    {
        "id": "ROUTE_KDY_ELL_BUS_01",
        "origin": "Kandy",
        "destination": "Ella",
        "transport_type": "Bus",
        "route": "Route 10 / Route 23 (Kandy - Nuwara Eliya - Bandarawela - Ella)",
        "service_operator": "SLTB / Private Provincial Coaches",
        "service_name": "Direct Highland Coach / Transfer via Bandarawela or Nuwara Eliya",
        "bus_or_train_number": "Route 10 (Kandy - Kataragama via Ella/Bandarawela)",
        "departure_location": "Goodshed Bus Stand (Kandy)",
        "arrival_location": "Ella Town Bus Stop / Main Road",
        "schedule_info": "Direct long-distance buses depart at 06:45 AM, 08:30 AM, and 12:15 PM; alternative route transfers at Nuwara Eliya or Bandarawela operate every 45 minutes",
        "estimated_duration": "4 hours 30 minutes to 5 hours 30 minutes",
        "estimated_distance_km": 135,
        "frequency": "3 direct daily buses, plus frequent connecting options via Bandarawela",
        "cost_estimate": "SLTB / Private Bus: approx LKR 500 – 750",
        "availability_status": "Daily public service in prototype knowledge base. Walk-in boarding.",
        "timing_feasibility_notes": "Shorter duration than train, but winding highland roads with sharp bends.",
        "notes": "Useful alternative when train tickets are fully booked.",
        "source": "prototype knowledge base",
        "last_updated": "2026-03-01"
    },
    {
        "id": "ROUTE_COL_GAL_TRAIN_01",
        "origin": "Colombo",
        "destination": "Galle",
        "transport_type": "Train",
        "route": "Coastal Line (Colombo Fort - Galle - Matara - Beliatta)",
        "service_operator": "Sri Lanka Railways",
        "service_name": "Coastal Express (Galu Kumari, Samudra Devi, Ruhunu Kumari)",
        "bus_or_train_number": "Train 8056 (Galu Kumari) / Train 8058 / Train 8085",
        "departure_location": "Colombo Fort Railway Station",
        "arrival_location": "Galle Railway Station",
        "schedule_info": "Daily departures: 06:50 AM (Coastal Express), 08:35 AM (Slow/Semi-Express), 10:30 AM, 02:25 PM, 03:50 PM (Samudra Devi), 05:30 PM (Ruhunu Kumari)",
        "estimated_duration": "2 hours to 2 hours 30 minutes",
        "estimated_distance_km": 119,
        "frequency": "7 to 10 scheduled daily coastal trains",
        "cost_estimate": "2nd Class: approx LKR 400 – 600; 3rd Class: approx LKR 200 – 300",
        "availability_status": "Regular daily service in prototype knowledge base. Real-time availability must be verified at stations.",
        "timing_feasibility_notes": "Early train arriving Galle before 09:30 AM allows plenty of daytime to explore Galle Dutch Fort ramparts before noon heat.",
        "notes": "Picturesque route traveling directly along the Indian Ocean coastline.",
        "source": "prototype knowledge base",
        "last_updated": "2026-03-01"
    },
    {
        "id": "ROUTE_COL_GAL_BUS_01",
        "origin": "Colombo",
        "destination": "Galle",
        "transport_type": "Bus",
        "route": "Southern Expressway (E01) Highway Express",
        "service_operator": "SLTB & Private Luxury Expressway Services",
        "service_name": "Expressway Luxury AC Bus",
        "bus_or_train_number": "Route EX1-001 (Colombo / Makumbura / Maharagama - Galle)",
        "departure_location": "Makumbura Multimodal Transport Hub (Kottawa) / Bastian Mawatha Pettah",
        "arrival_location": "Galle Central Bus Stand",
        "schedule_info": "Frequent expressway service departing every 20 minutes from 05:30 AM to 09:30 PM daily",
        "estimated_duration": "1 hour 15 minutes to 1 hour 45 minutes (via E01 expressway)",
        "estimated_distance_km": 115,
        "frequency": "Every 20 minutes",
        "cost_estimate": "Expressway AC: approx LKR 750 – 1,000",
        "availability_status": "High walk-in frequency. Seats allocated upon arrival at terminal.",
        "timing_feasibility_notes": "Fastest transport mode between Colombo and Galle. Ideal for day trips.",
        "notes": "Non-stop air-conditioned transit via Southern Expressway.",
        "source": "prototype knowledge base",
        "last_updated": "2026-03-01"
    },
    {
        "id": "ROUTE_KDY_NUR_BUS_01",
        "origin": "Kandy",
        "destination": "Nuwara Eliya",
        "transport_type": "Bus",
        "route": "Route 47 / Route 79 (Kandy - Nuwara Eliya via Gampola & Ramboda Pass A5)",
        "service_operator": "SLTB & Private Bus Operators",
        "service_name": "Highland Regular / AC Express Bus",
        "bus_or_train_number": "Route 47",
        "departure_location": "Goodshed Bus Stand (Kandy)",
        "arrival_location": "Nuwara Eliya Central Bus Stand",
        "schedule_info": "Regular departures every 30 minutes from 05:30 AM to 06:30 PM",
        "estimated_duration": "2 hours 30 minutes to 3 hours 15 minutes",
        "estimated_distance_km": 75,
        "frequency": "Every 30 minutes",
        "cost_estimate": "Normal bus: approx LKR 250 – 350; AC mini-bus: approx LKR 600",
        "availability_status": "Continuous walk-in availability in prototype knowledge base.",
        "timing_feasibility_notes": "Arrive Nuwara Eliya by mid-day for cooler climate afternoon activities around Lake Gregory.",
        "notes": "Passes spectacular mountain scenery and Ramboda Falls along the winding A5 highway.",
        "source": "prototype knowledge base",
        "last_updated": "2026-03-01"
    },
    {
        "id": "ROUTE_KDY_NUR_TRAIN_01",
        "origin": "Kandy",
        "destination": "Nuwara Eliya",
        "transport_type": "Train",
        "route": "Main Line (Kandy / Peradeniya - Nanu Oya Station + road transfer)",
        "service_operator": "Sri Lanka Railways + Local Taxi/Bus",
        "service_name": "Highland Express Train to Nanu Oya",
        "bus_or_train_number": "Train 1005 / Train 1015 to Nanu Oya",
        "departure_location": "Kandy Railway Station / Peradeniya Junction",
        "arrival_location": "Nanu Oya Railway Station (9 km from Nuwara Eliya town)",
        "schedule_info": "Morning trains at 08:47 AM (from Peradeniya) and 11:10 AM; arrives Nanu Oya at approx 12:40 PM and 03:15 PM",
        "estimated_duration": "Train: 3 hours 45 mins; Connecting taxi/bus from Nanu Oya to Nuwara Eliya: 20-30 mins",
        "estimated_distance_km": 88,
        "frequency": "3 to 4 scheduled daily services",
        "cost_estimate": "Train 2nd Class: LKR 600; Connecting Tuk-Tuk to Nuwara Eliya town: LKR 1,000 – 1,500",
        "availability_status": "Scheduled daily operations in prototype knowledge base. Note that Nanu Oya is the rail gateway for Nuwara Eliya.",
        "timing_feasibility_notes": "Must factor in 30 minutes for taxi or local feeder bus transfer from Nanu Oya station up to Nuwara Eliya town center.",
        "notes": "Nuwara Eliya town does not have a direct railway station; Nanu Oya is the designated gateway station.",
        "source": "prototype knowledge base",
        "last_updated": "2026-03-01"
    }
]

print(f"✅ Defined prototype logistics knowledge base with {len(PROTOTYPE_LOGISTICS_DATA)} verified transit corridors.")

✅ Defined prototype logistics knowledge base with 8 verified transit corridors.


---
## Section 8: Create LangChain Document Objects

### What we are building:
Convert the raw dictionary records into LangChain `Document` objects containing structured text in `page_content` and filtering metadata.

### Why we need it:
Standard LangChain RAG pipelines operate on `Document` objects, preserving source provenance and routing attributes in metadata.

In [6]:
# Section 8: Create LangChain Document Objects
logistics_documents: List[Document] = []

for item in PROTOTYPE_LOGISTICS_DATA:
    content = (
        f"Origin: {item['origin']}\n"
        f"Destination: {item['destination']}\n"
        f"Transport Type: {item['transport_type']}\n"
        f"Route Name/Code: {item['route']}\n"
        f"Service/Bus/Train Number: {item['bus_or_train_number']}\n"
        f"Service Name: {item['service_name']}\n"
        f"Operator: {item['service_operator']}\n"
        f"Departure Station/Location: {item['departure_location']}\n"
        f"Arrival Station/Location: {item['arrival_location']}\n"
        f"Schedule Information: {item['schedule_info']}\n"
        f"Estimated Journey Duration: {item['estimated_duration']}\n"
        f"Approximate Distance: {item['estimated_distance_km']} km\n"
        f"Frequency: {item['frequency']}\n"
        f"Estimated Cost: {item['cost_estimate']}\n"
        f"Availability Status: {item['availability_status']}\n"
        f"Timing Feasibility Notes: {item['timing_feasibility_notes']}\n"
        f"Important Notes: {item['notes']}\n"
        f"Knowledge Base Source: {item['source']}\n"
        f"Last Updated: {item['last_updated']}"
    )
    doc = Document(
        page_content=content.strip(),
        metadata={
            "origin": item["origin"],
            "destination": item["destination"],
            "transport_type": item["transport_type"],
            "route": item["route"],
            "bus_or_train_number": item["bus_or_train_number"],
            "source": item["source"]
        }
    )
    logistics_documents.append(doc)

print(f"✅ Created {len(logistics_documents)} LangChain Document objects.")
print("\n--- Sample Document Preview ---")
print(logistics_documents[0].page_content[:280] + "...")

✅ Created 8 LangChain Document objects.

--- Sample Document Preview ---
Origin: Colombo
Destination: Kandy
Transport Type: Train
Route Name/Code: Main Line (Colombo Fort - Badulla / Kandy)
Service/Bus/Train Number: Train 1005 / Train 1015 / Train 1009
Service Name: Intercity & Express Trains (Podi Menike, Udarata Menike)
Operator: Sri Lanka Railways
...


---
## Section 9: Create Dense Semantic Embeddings

### What we are building:
Initialize `HuggingFaceEmbeddings` using `sentence-transformers/all-MiniLM-L6-v2`.

### Why we need it:
Converts text into 384-dimensional dense vectors representing conceptual meaning, enabling semantic search (e.g. matching 'train travel to the hill country' with 'Colombo to Kandy train').

In [7]:
# Section 9: Create Dense Semantic Embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    show_progress=False
)
print("✅ Initialized HuggingFaceEmbeddings ('sentence-transformers/all-MiniLM-L6-v2').")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Initialized HuggingFaceEmbeddings ('sentence-transformers/all-MiniLM-L6-v2').


---
## Section 10: Create FAISS Vector Store

### What we are building:
Chunk documents with `RecursiveCharacterTextSplitter` and index them into an in-memory `FAISS` vector store.

### Why we need it:
Fast, sub-millisecond semantic similarity search over document chunks.

In [8]:
# Section 10: Create FAISS Vector Store
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=70,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunked_logistics_docs = text_splitter.split_documents(logistics_documents)
print(f"✅ Split {len(logistics_documents)} documents into {len(chunked_logistics_docs)} chunks.")

vectorstore = FAISS.from_documents(chunked_logistics_docs, embedding_model)
print("✅ Successfully indexed chunks into FAISS vector store.")

✅ Split 8 documents into 18 chunks.
✅ Successfully indexed chunks into FAISS vector store.


---
## Section 11: Create Retriever

### What we are building:
Convert the vector store into a LangChain `Retriever` configured to retrieve top `k=4` chunks.

### Why we need it:
The retriever acts as the factual grounding bridge between vector database and the agent's tools.

In [9]:
# Section 11: Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("✅ Configured FAISS Retriever with top_k=4.")

✅ Configured FAISS Retriever with top_k=4.


---
## Section 12: Test Retriever

### What we are building:
Validate the retriever with sample natural language transit queries.

### Why we need it:
Ensures semantic retrieval operates accurately before exposing it as an agent tool.

In [10]:
# Section 12: Test Retriever
test_query = "How to travel from Colombo to Kandy by train?"
retrieved_docs = retriever.invoke(test_query)

print(f"🔍 Test Query: '{test_query}'")
print(f"📄 Retrieved {len(retrieved_docs)} relevant chunk(s):\n")
for i, d in enumerate(retrieved_docs, start=1):
    print(f"--- [Retrieved Chunk {i}] Metadata: {d.metadata['route']} ({d.metadata['transport_type']}) ---")
    print(d.page_content[:200] + "...\n")

🔍 Test Query: 'How to travel from Colombo to Kandy by train?'
📄 Retrieved 4 relevant chunk(s):

--- [Retrieved Chunk 1] Metadata: Main Line (Colombo Fort - Badulla / Kandy) (Train) ---
Origin: Colombo
Destination: Kandy
Transport Type: Train
Route Name/Code: Main Line (Colombo Fort - Badulla / Kandy)
Service/Bus/Train Number: Train 1005 / Train 1015 / Train 1009
Service Name: Interc...

--- [Retrieved Chunk 2] Metadata: Route 01 (Colombo - Kandy via A1 Highway) (Bus) ---
Origin: Colombo
Destination: Kandy
Transport Type: Bus
Route Name/Code: Route 01 (Colombo - Kandy via A1 Highway)
Service/Bus/Train Number: Bus Route 01
Service Name: Route 01 Semi-Luxury / AC Interci...

--- [Retrieved Chunk 3] Metadata: Coastal Line (Colombo Fort - Galle - Matara - Beliatta) (Train) ---
Origin: Colombo
Destination: Galle
Transport Type: Train
Route Name/Code: Coastal Line (Colombo Fort - Galle - Matara - Beliatta)
Service/Bus/Train Number: Train 8056 (Galu Kumari) / Train 8058 / Trai...

--- [Retriev

---
## Section 13: Create Logistics Tools & Clean Service Abstractions

### What we are building:
We create specialized LangChain tools backed by clean service abstractions:
1. `search_logistics_information`: RAG semantic search over transit corridors.
2. `search_transport_routes`: Route configurations, distances, and modes.
3. `search_transport_schedule`: Departure times and frequencies.
4. `check_transport_availability`: Availability guidance with strict live-data guardrail.
5. `search_attraction_hours`: Opening hours for timing feasibility.

### Why we need it:
The service abstractions (`TransportService`, `RouteService`, `AvailabilityService`) decouple prototype storage from future live external APIs.

In [11]:
# Section 13: Create Logistics Tools & Clean Service Abstractions

# 1. Transport Service Abstraction
class TransportService:
    """Abstract service for querying transport options & timetables."""
    def __init__(self, data=None):
        self.data = data or PROTOTYPE_LOGISTICS_DATA

    def get_schedule(self, origin: str, destination: str, transport_type: Optional[str] = None):
        orig_clean = origin.strip().lower()
        dest_clean = destination.strip().lower()
        type_clean = transport_type.strip().lower() if transport_type else None

        matched = []
        for r in self.data:
            r_orig = r.get("origin", "").lower()
            r_dest = r.get("destination", "").lower()
            r_type = r.get("transport_type", "").lower()
            if (orig_clean in r_orig or r_orig in orig_clean) and (dest_clean in r_dest or r_dest in dest_clean):
                if not type_clean or type_clean in r_type or r_type in type_clean:
                    matched.append(r)
        return matched

# 2. Route Service Abstraction
class RouteService:
    """Abstract service for routing and distance evaluation."""
    def __init__(self, data=None):
        self.data = data or PROTOTYPE_LOGISTICS_DATA

    def find_routes(self, origin: str, destination: str):
        orig_clean = origin.strip().lower()
        dest_clean = destination.strip().lower()
        matched = []
        for r in self.data:
            r_orig = r.get("origin", "").lower()
            r_dest = r.get("destination", "").lower()
            if (orig_clean in r_orig or r_orig in orig_clean) and (dest_clean in r_dest or r_dest in dest_clean):
                matched.append(r)
        return matched

# 3. Availability Service Abstraction
class AvailabilityService:
    """Abstract service for availability checks and timing feasibility."""
    def __init__(self, data=None):
        self.data = data or PROTOTYPE_LOGISTICS_DATA

    def check_availability(self, origin: str, destination: str, transport_type: Optional[str] = None, travel_date: Optional[str] = None):
        orig_clean = origin.strip().lower()
        dest_clean = destination.strip().lower()
        type_clean = transport_type.strip().lower() if transport_type else None

        matched = []
        for r in self.data:
            r_orig = r.get("origin", "").lower()
            r_dest = r.get("destination", "").lower()
            r_type = r.get("transport_type", "").lower()
            if (orig_clean in r_orig or r_orig in orig_clean) and (dest_clean in r_dest or r_dest in dest_clean):
                if not type_clean or type_clean in r_type or r_type in type_clean:
                    matched.append(r)
        return {
            "matched_records": matched,
            "has_date_query": bool(travel_date and travel_date.strip()),
            "travel_date": travel_date
        }

transport_service = TransportService(PROTOTYPE_LOGISTICS_DATA)
route_service = RouteService(PROTOTYPE_LOGISTICS_DATA)
availability_service = AvailabilityService(PROTOTYPE_LOGISTICS_DATA)

# 4. Tool Schemas & Factory Definitions
class LogisticsSearchInput(BaseModel):
    query: str = Field(description="Search query describing origin, destination, transport type, route, or schedule.")

@tool(args_schema=LogisticsSearchInput)
def search_logistics_information(query: str) -> str:
    """
    Search the prototype tourism logistics knowledge base for factual transport options,
    public transit routes, bus numbers, train routes, schedules, journey durations,
    and availability between Sri Lankan destinations.
    """
    if not query or not query.strip():
        return "ERROR: Search query is empty. Please provide a valid transit inquiry."
    clean_query = query.strip()
    try:
        results = retriever.invoke(clean_query)
    except Exception as e:
        return f"TOOL_ERROR: Logistics retriever encountered an error: {str(e)}"

    if not results:
        return f"NO_INFORMATION_FOUND: No relevant logistics information found for '{clean_query}' in the prototype knowledge base."

    formatted = []
    for i, doc in enumerate(results, start=1):
        formatted.append(f"--- [Logistics Record {i}] ---\n{doc.page_content}\nMetadata: {json.dumps(doc.metadata)}")
    return "\n\n".join(formatted)

class RouteSearchInput(BaseModel):
    origin: str = Field(description="Origin city or location (e.g. 'Colombo', 'Kandy')")
    destination: str = Field(description="Destination city or location (e.g. 'Kandy', 'Ella', 'Galle')")

@tool(args_schema=RouteSearchInput)
def search_transport_routes(origin: str, destination: str) -> str:
    """Search for practical transit routes, distances, and duration comparisons between two Sri Lankan locations."""
    if not origin or not origin.strip() or not destination or not destination.strip():
        return "ERROR: Both origin and destination must be non-empty strings."
    records = route_service.find_routes(origin, destination)
    if not records:
        return f"NO_ROUTE_FOUND: No practical transit routes between '{origin}' and '{destination}' were found in the prototype knowledge base."
    lines = [f"Found {len(records)} route option(s) connecting {origin} and {destination}:"]
    for idx, rec in enumerate(records, start=1):
        lines.append(
            f"\nOption {idx}: [{rec.get('transport_type')}] {rec.get('route')}"
            f"\n  • Identifier: {rec.get('bus_or_train_number')}"
            f"\n  • Approx Distance: {rec.get('estimated_distance_km')} km"
            f"\n  • Approx Travel Duration: {rec.get('estimated_duration')}"
            f"\n  • Departure: {rec.get('departure_location')}"
            f"\n  • Arrival: {rec.get('arrival_location')}"
            f"\n  • Source: {rec.get('source')}"
        )
    return "\n".join(lines)

class TransportScheduleInput(BaseModel):
    origin: str = Field(description="Origin city or station")
    destination: str = Field(description="Destination city or station")
    transport_type: Optional[str] = Field(default=None, description="Optional filter: 'Train' or 'Bus'")

@tool(args_schema=TransportScheduleInput)
def search_transport_schedule(origin: str, destination: str, transport_type: Optional[str] = None) -> str:
    """Retrieve scheduled departure times, service names, and frequencies for trains or buses."""
    if not origin or not origin.strip() or not destination or not destination.strip():
        return "ERROR: Both origin and destination must be provided."
    records = transport_service.get_schedule(origin, destination, transport_type)
    if not records:
        return f"NO_SCHEDULE_FOUND: No scheduled {transport_type or 'transport'} services found between {origin} and {destination}."
    lines = [f"Found {len(records)} prototype schedule entry/entries for {origin} to {destination}:"]
    for idx, rec in enumerate(records, start=1):
        lines.append(
            f"\n[{idx}] {rec.get('transport_type')} - {rec.get('service_name')}"
            f"\n  • Route / Code: {rec.get('route')} ({rec.get('bus_or_train_number')})"
            f"\n  • Schedule Info: {rec.get('schedule_info')}"
            f"\n  • Frequency: {rec.get('frequency')}"
            f"\n  • Duration: {rec.get('estimated_duration')}"
            f"\n  • Availability Status: {rec.get('availability_status')}"
            f"\n  • Source: {rec.get('source')}"
        )
    return "\n".join(lines)

class TransportAvailabilityInput(BaseModel):
    origin: str = Field(description="Origin location")
    destination: str = Field(description="Destination location")
    transport_type: Optional[str] = Field(default=None, description="Optional transport mode")
    travel_date: Optional[str] = Field(default=None, description="Optional travel date or timeframe")

@tool(args_schema=TransportAvailabilityInput)
def check_transport_availability(origin: str, destination: str, transport_type: Optional[str] = None, travel_date: Optional[str] = None) -> str:
    """Check transport availability between two locations based on prototype knowledge base."""
    if not origin or not origin.strip() or not destination or not destination.strip():
        return "ERROR: Both origin and destination are required."
    res = availability_service.check_availability(origin, destination, transport_type, travel_date)
    records = res["matched_records"]
    if not records:
        return f"NO_AVAILABILITY_DATA: No transport availability data for '{origin}' to '{destination}' in prototype knowledge base."
    lines = [f"Availability status for {origin} to {destination} ({transport_type or 'All Modes'}):"]
    if res["has_date_query"]:
        lines.append(
            f"\n⚠️ LIVE DATA LIMITATION: Real-time operational availability for specific dates/times ('{travel_date}') "
            f"cannot be verified through this prototype. Live reservation APIs are not connected."
        )
    for idx, rec in enumerate(records, start=1):
        lines.append(
            f"\n[{idx}] {rec.get('transport_type')} - {rec.get('service_name')}"
            f"\n  • Status: {rec.get('availability_status')}"
            f"\n  • Typical Schedule: {rec.get('schedule_info')}"
            f"\n  • Booking Guidance: {rec.get('notes')}"
        )
    lines.append("\nNote: Booking is NOT currently supported. The system cannot reserve tickets.")
    return "\n".join(lines)

class AttractionHoursInput(BaseModel):
    attraction_name: str = Field(description="Name of the tourist attraction")
    destination: Optional[str] = Field(default=None, description="Optional destination city")

@tool(args_schema=AttractionHoursInput)
def search_attraction_hours(attraction_name: str, destination: Optional[str] = None) -> str:
    """Retrieve opening and closing hours for an attraction to evaluate travel timing feasibility."""
    if not attraction_name or not attraction_name.strip():
        return "ERROR: Attraction name must be provided."
    known_attractions = {
        "temple of the tooth": "Temple of the Sacred Tooth Relic (Kandy) | Opening Hours: 5:30 AM – 8:00 PM daily | Duration: 2 to 3 hours",
        "nine arches bridge": "Nine Arches Bridge (Ella) | Opening Hours: Open 24 hours (daylight recommended) | Duration: 1 to 2 hours",
        "galle fort": "Galle Fort Ramparts (Galle) | Opening Hours: Open 24 hours daily | Duration: 3 to 4 hours"
    }
    query_clean = attraction_name.lower().strip()
    for k, v in known_attractions.items():
        if k in query_clean or query_clean in k:
            return f"Timing feasibility record:\n{v}"
    return f"NO_TIMING_DATA: No verified opening hours found for '{attraction_name}'. State that opening hours could not be verified."

agent_tools = [
    search_logistics_information,
    search_transport_routes,
    search_transport_schedule,
    check_transport_availability,
    search_attraction_hours
]

print(f"✅ Created {len(agent_tools)} specialized LangChain logistics tools directly in notebook:")
for t in agent_tools:
    print(f"   • {t.name}")


✅ Created 5 specialized LangChain logistics tools directly in notebook:
   • search_logistics_information
   • search_transport_routes
   • search_transport_schedule
   • check_transport_availability
   • search_attraction_hours


---
## Section 14: Define Pydantic Input Schema

### What we are building:
Define `LogisticsRequest` with optional filters (`transport_type`, `travel_date`, `preferred_time`).

### Why we need it:
Runtime type checking prevents malformed inputs from reaching the agent workflow.

In [12]:
# Section 14: Define Pydantic Input Schema
class LogisticsRequest(BaseModel):
    """Structured input schema for logistics inquiries."""
    origin: str = Field(description="Starting city, station, or hub (e.g. 'Colombo', 'Kandy')")
    destination: str = Field(description="Target destination (e.g. 'Kandy', 'Ella', 'Galle')")
    transport_type: Optional[str] = Field(default=None, description="Optional mode: 'Train', 'Bus', 'Highway Express Bus'")
    travel_date: Optional[str] = Field(default=None, description="Optional travel date/timeframe (e.g. 'tomorrow morning')")
    preferred_time: Optional[str] = Field(default=None, description="Optional preferred time of day (e.g. 'morning')")
    requested_information: Optional[str] = Field(default=None, description="Specific inquiry or focus")

# Test validation
req = LogisticsRequest(origin="Colombo", destination="Kandy", transport_type="Train")
print(f"✅ Validated sample request: {req.model_dump()}")

✅ Validated sample request: {'origin': 'Colombo', 'destination': 'Kandy', 'transport_type': 'Train', 'travel_date': None, 'preferred_time': None, 'requested_information': None}


---
## Section 15: Define Pydantic Output Schema

### What we are building:
Define `TransportOption` and `LogisticsResult` models with `Field(default_factory=list)`.

### Why we need it:
Guarantees a strict, predictable JSON interface that the Travel Planning Agent can parse directly.

In [13]:
# Section 15: Define Pydantic Output Schema
class TransportOption(BaseModel):
    """Structured representation of a single transport option."""
    transport_type: str = Field(description="Mode of transit: 'Train', 'Bus', 'Highway Express Bus', 'Taxi'")
    route: str = Field(description="Route name, highway code, or rail line")
    departure_location: Optional[str] = Field(default=None, description="Departure terminal or station")
    arrival_location: Optional[str] = Field(default=None, description="Arrival terminal or station")
    departure_time: Optional[str] = Field(default=None, description="Scheduled departure times")
    arrival_time: Optional[str] = Field(default=None, description="Estimated arrival time or window")
    estimated_duration: Optional[str] = Field(default=None, description="Estimated transit duration")
    availability: Optional[str] = Field(default=None, description="Availability guidance or seat booking status")
    source: Optional[str] = Field(default="prototype knowledge base", description="Data provenance")


class LogisticsResult(BaseModel):
    """Structured output schema returned by the agent."""
    origin: str = Field(description="Origin location")
    destination: str = Field(description="Destination location")
    travel_date: Optional[str] = Field(default=None, description="Requested travel date")
    transport_options: List[TransportOption] = Field(default_factory=list, description="Verified transit options")
    information_limitations: List[str] = Field(default_factory=list, description="Missing or unverified data notices")
    unsupported_requests_note: Optional[str] = Field(default=None, description="Boundary notices (e.g. itinerary, booking)")

print("✅ Defined Pydantic output schemas (TransportOption, LogisticsResult).")

✅ Defined Pydantic output schemas (TransportOption, LogisticsResult).


---
## Section 16: Define Agent State

### What we are building:
Define `LogisticsAgentState` using Python `TypedDict` and LangGraph's `add_messages` reducer.

### Why we need it:
LangGraph tracks agent execution across graph nodes using state.

In [14]:
# Section 16: Define Agent State
class LogisticsAgentState(TypedDict):
    """State tracked across the LangGraph agent loop."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    current_request: Dict[str, Any]
    tool_results: List[str]
    errors: List[str]
    execution_step_count: int
    final_result: Optional[Dict[str, Any]]

print("✅ Defined LogisticsAgentState.")

✅ Defined LogisticsAgentState.


---
## Section 17: Create Agent System Prompt

### What we are building:
Define `SYSTEM_PROMPT` containing identity, task instructions, strict boundary guardrails, grounding rules, and JSON schema requirements.

### Why we need it:
The system prompt enforces behavioral safety bounds: refusing to create itineraries, refusing to make personalized recommendations, and disclaiming ticket bookings.

In [15]:
# Section 17: Create Agent System Prompt
SYSTEM_PROMPT = """You are the NOVA Travel Logistics & Availability Agent for the NOVA Smart Tourism Platform in Sri Lanka.

YOUR MISSION:
Provide factual travel logistics, public transit routes, timetables, travel durations, and availability information.

WHAT YOU RESEARCH:
- Transport options (Train, Bus, Highway Express Bus, Taxi)
- Public transit details: bus numbers, train names/numbers, departure/arrival stations
- Schedules, departure frequencies, and travel durations
- Approximate travel distances
- Transport availability guidance
- Attraction opening/closing hours for travel feasibility

STRICT OPERATIONAL RULES:
1. ALWAYS USE AVAILABLE TOOLS: Call your tools (`search_logistics_information`, `search_transport_routes`, `search_transport_schedule`, `check_transport_availability`, `search_attraction_hours`) whenever factual transit data is required.
2. RETRIEVAL GROUNDING: Rely strictly on retrieved facts. NEVER fabricate routes, schedules, prices, or seat availability.
3. PROTOTYPE ATTRIBUTION: All data is from a prototype knowledge base. Clearly state that live, real-time seat availability cannot be verified without official live API connections.
4. STRICT RESPONSIBILITY BOUNDARIES:
   - DO NOT CREATE ITINERARIES: Explain: "Itinerary generation and daily scheduling are handled by the Travel Planning Agent. As the Travel Logistics & Availability Agent, I can provide the transit routes, travel durations, and schedules required for your destinations."
   - DO NOT RECOMMEND DESTINATIONS: Explain: "Personalized destination recommendations, review sentiment analysis, and suitability scoring are handled by the Recommendation & Feedback Analysis Agent."
   - DO NOT PERFORM BOOKINGS: Explain: "Ticket booking functionality is not currently connected to live booking engines. I can provide schedule and availability guidance, but cannot process reservations."
5. INFORMATION LIMITATIONS: If no route, schedule, or availability data exists, list it in `information_limitations`.

STRUCTURED OUTPUT FORMAT:
Conclude your final response with valid JSON adhering to this schema:
```json
{
  "origin": "<Origin>",
  "destination": "<Destination>",
  "travel_date": "<Date or null>",
  "transport_options": [
    {
      "transport_type": "Train / Bus / etc.",
      "route": "...",
      "departure_location": "...",
      "arrival_location": "...",
      "departure_time": "...",
      "arrival_time": "...",
      "estimated_duration": "...",
      "availability": "...",
      "source": "prototype knowledge base"
    }
  ],
  "information_limitations": ["..."],
  "unsupported_requests_note": "<Optional note or null>"
}
```
"""

print("✅ Agent system prompt configured with strict single-responsibility boundaries.")

✅ Agent system prompt configured with strict single-responsibility boundaries.


---
## Section 18: Build LangGraph Agent Workflow

### What we are building:
Construct the LangGraph `StateGraph` linking `agent_node`, `tools_node`, and `validation_node`.

### Why we need it:
Implements the autonomous **Think → Act → Observe** agentic loop with bounded execution safety.

In [16]:
# Section 18: Build LangGraph Agent Workflow

MAX_STEPS = 5  # Bounded execution guardrail

class SafeExecutionTracer:
    """Operational event logger recording tool executions and validation checkpoints."""
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
        self.logs: List[Dict[str, Any]] = []

    def log_step(self, step: int, action: str, details: Dict[str, Any]):
        entry = {
            "agent": "Travel Logistics & Availability Agent",
            "step": step,
            "action": action,
            **details
        }
        self.logs.append(entry)
        if self.verbose:
            print(f"\n[AGENT TRACE | Step {step}] Action: {action}")
            for k, v in details.items():
                print(f"   • {k}: {v}")

    def get_logs(self) -> List[Dict[str, Any]]:
        return self.logs


class TravelLogisticsAgent:
    """Encapsulates the Travel Logistics & Availability Agent LangGraph workflow."""
    def __init__(
        self,
        model_name: str = "gemini-3.5-flash-lite",
        temperature: float = 0.0,
        retriever=None,
        tracer: Optional[SafeExecutionTracer] = None
    ):
        self.model_name = model_name
        self.temperature = temperature
        self.tracer = tracer or SafeExecutionTracer(verbose=False)
        self.retriever = retriever

        self.tools = agent_tools
        self.tools_by_name = {t.name: t for t in self.tools}

        self.llm = ChatGoogleGenerativeAI(
            model=self.model_name,
            temperature=self.temperature
        )
        self.llm_with_tools = self.llm.bind_tools(self.tools)
        self.app = self._build_graph()

    def _agent_node(self, state: LogisticsAgentState) -> Dict[str, Any]:
        step_count = state.get("execution_step_count", 0) + 1

        # Bounded Execution Check (Safety Guardrail)
        if step_count > MAX_STEPS:
            error_msg = f"Bounded execution limit reached (MAX_STEPS={MAX_STEPS}). Halting loop safely."
            self.tracer.log_step(
                step=step_count,
                action="Bounded Execution Limit Triggered",
                details={"status": "Halted", "message": error_msg}
            )
            fallback_msg = AIMessage(
                content=json.dumps({
                    "origin": state.get("current_request", {}).get("origin", "Unknown"),
                    "destination": state.get("current_request", {}).get("destination", "Unknown"),
                    "travel_date": state.get("current_request", {}).get("travel_date"),
                    "transport_options": [],
                    "information_limitations": [
                        "Unable to complete the logistics request within the allowed number of steps.",
                        "Execution halted to protect agentic safety bounds."
                    ],
                    "unsupported_requests_note": error_msg
                })
            )
            return {
                "messages": [fallback_msg],
                "errors": state.get("errors", []) + [error_msg],
                "execution_step_count": step_count
            }

        messages = state["messages"]
        response = self.llm_with_tools.invoke(messages)

        has_calls = bool(hasattr(response, "tool_calls") and response.tool_calls)
        self.tracer.log_step(
            step=step_count,
            action="LLM Reasoning Step",
            details={
                "has_tool_calls": has_calls,
                "tool_calls_requested": [c["name"] for c in response.tool_calls] if has_calls else []
            }
        )

        return {
            "messages": [response],
            "execution_step_count": step_count
        }

    def _tools_node(self, state: LogisticsAgentState) -> Dict[str, Any]:
        last_message = state["messages"][-1]
        tool_messages = []
        tool_results = list(state.get("tool_results", []))
        errors = list(state.get("errors", []))
        step_count = state.get("execution_step_count", 1)

        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            for tool_call in last_message.tool_calls:
                tool_name = tool_call["name"]
                tool_args = tool_call["args"]
                tool_id = tool_call["id"]

                if tool_name in self.tools_by_name:
                    try:
                        observation = self.tools_by_name[tool_name].invoke(tool_args)
                        self.tracer.log_step(
                            step=step_count,
                            action="Tool Executed Successfully",
                            details={
                                "tool": tool_name,
                                "args": tool_args,
                                "observation_preview": str(observation)[:120] + "..."
                            }
                        )
                    except Exception as exc:
                        observation = f"TOOL_ERROR: Execution of {tool_name} failed: {str(exc)}"
                        errors.append(observation)
                else:
                    observation = f"TOOL_ERROR: Unknown tool '{tool_name}' requested."
                    errors.append(observation)

                tool_messages.append(
                    ToolMessage(
                        content=str(observation),
                        tool_call_id=tool_id,
                        name=tool_name
                    )
                )
                tool_results.append(f"[{tool_name}] args={tool_args}")

        return {
            "messages": tool_messages,
            "tool_results": tool_results,
            "errors": errors
        }

    def _validation_node(self, state: LogisticsAgentState) -> Dict[str, Any]:
        last_message = state["messages"][-1]
        content = last_message.content if hasattr(last_message, "content") else str(last_message)
        req = state.get("current_request", {})
        origin_name = req.get("origin", "Unknown")
        dest_name = req.get("destination", "Unknown")
        travel_date = req.get("travel_date")

        raw_text = ""
        if isinstance(content, str):
            raw_text = content
        elif isinstance(content, list):
            texts = [c.get("text", "") for c in content if isinstance(c, dict) and "text" in c]
            raw_text = "\n".join(texts)

        clean_json = raw_text.strip()
        if "```json" in clean_json:
            clean_json = clean_json.split("```json")[1].split("```")[0].strip()
        elif "```" in clean_json:
            clean_json = clean_json.split("```")[1].split("```")[0].strip()

        try:
            data = json.loads(clean_json)
            validated = LogisticsResult(**data)
            parsed_result = validated.model_dump()
            self.tracer.log_step(
                step=state.get("execution_step_count", 0),
                action="Output Validation Passed",
                details={"transport_options_count": len(validated.transport_options)}
            )
        except (json.JSONDecodeError, ValidationError) as err:
            unsupported_note = None
            raw_lower = raw_text.lower()
            if "travel planning agent" in raw_lower or "itinerary" in raw_lower:
                unsupported_note = "Itinerary generation is handled by the Travel Planning Agent."
            elif "recommendation" in raw_lower or "feedback analysis" in raw_lower:
                unsupported_note = "Destination recommendations are handled by the Recommendation & Feedback Analysis Agent."
            elif "booking" in raw_lower or "ticket" in raw_lower:
                unsupported_note = "Ticket booking functionality is not currently connected."

            parsed_result = LogisticsResult(
                origin=origin_name,
                destination=dest_name,
                travel_date=travel_date,
                transport_options=[],
                information_limitations=[
                    "Response structured according to safety fallback schema.",
                    f"Summary message: {raw_text[:280]}"
                ],
                unsupported_requests_note=unsupported_note or f"Validation fallback: {str(err)}"
            ).model_dump()

        return {"final_result": parsed_result}

    def _route_tools(self, state: LogisticsAgentState) -> str:
        last_message = state["messages"][-1]
        step_count = state.get("execution_step_count", 0)

        if step_count >= MAX_STEPS:
            return "validation_node"

        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            return "tools_node"

        return "validation_node"

    def _build_graph(self):
        graph = StateGraph(LogisticsAgentState)
        graph.add_node("agent_node", self._agent_node)
        graph.add_node("tools_node", self._tools_node)
        graph.add_node("validation_node", self._validation_node)

        graph.add_edge(START, "agent_node")
        graph.add_conditional_edges(
            "agent_node",
            self._route_tools,
            {
                "tools_node": "tools_node",
                "validation_node": "validation_node"
            }
        )
        graph.add_edge("tools_node", "agent_node")
        graph.add_edge("validation_node", END)

        return graph.compile()

    def run(self, request_input: Any) -> Dict[str, Any]:
        if not request_input:
            err_result = LogisticsResult(
                origin="Unknown",
                destination="Unknown",
                transport_options=[],
                information_limitations=["ERROR: Input request is empty. Please provide a valid transit inquiry."],
                unsupported_requests_note="Input Validation Error: Empty Query"
            ).model_dump()
            return {"result": err_result, "tool_results": [], "errors": ["Input query is empty."], "step_count": 0}

        if isinstance(request_input, str):
            clean_str = request_input.strip()
            if not clean_str:
                err_result = LogisticsResult(
                    origin="Unknown",
                    destination="Unknown",
                    transport_options=[],
                    information_limitations=["ERROR: Input query string is blank."],
                    unsupported_requests_note="Input Validation Error: Empty Query"
                ).model_dump()
                return {"result": err_result, "tool_results": [], "errors": ["Input query string is blank."], "step_count": 0}
            request_data = {"origin": "Inferred from query", "destination": "Inferred from query", "requested_information": clean_str}
            user_prompt = clean_str
        elif isinstance(request_input, LogisticsRequest):
            request_data = request_input.model_dump()
            user_prompt = f"Logistics Inquiry: Travel from {request_data['origin']} to {request_data['destination']}."
            if request_data.get("transport_type"):
                user_prompt += f" Preferred mode: {request_data['transport_type']}."
            if request_data.get("travel_date"):
                user_prompt += f" Travel date: {request_data['travel_date']}."
        elif isinstance(request_input, dict):
            try:
                validated_req = LogisticsRequest(**request_input)
                request_data = validated_req.model_dump()
                user_prompt = f"Logistics Inquiry: Travel from {request_data['origin']} to {request_data['destination']}."
            except ValidationError as ve:
                err_result = LogisticsResult(
                    origin=request_input.get("origin", "Unknown"),
                    destination=request_input.get("destination", "Unknown"),
                    transport_options=[],
                    information_limitations=[f"ERROR: Input validation failed: {str(ve)}"],
                    unsupported_requests_note="Pydantic Input Validation Error"
                ).model_dump()
                return {"result": err_result, "tool_results": [], "errors": [str(ve)], "step_count": 0}

        initial_state: LogisticsAgentState = {
            "messages": [
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=user_prompt)
            ],
            "current_request": request_data,
            "tool_results": [],
            "errors": [],
            "execution_step_count": 0,
            "final_result": None
        }

        result_state = self.app.invoke(initial_state)
        return {
            "result": result_state.get("final_result"),
            "tool_results": result_state.get("tool_results", []),
            "errors": result_state.get("errors", []),
            "step_count": result_state.get("execution_step_count", 0),
            "trace": self.tracer.get_logs()
        }

# Instantiate agent
tracer = SafeExecutionTracer(verbose=False)
logistics_agent = TravelLogisticsAgent(
    model_name="gemini-3.5-flash-lite",
    temperature=0.0,
    retriever=retriever,
    tracer=tracer
)

print(f"✅ TravelLogisticsAgent successfully instantiated with LangGraph workflow (MAX_STEPS={MAX_STEPS}).")


✅ TravelLogisticsAgent successfully instantiated with LangGraph workflow (MAX_STEPS=5).


---
## Section 19: Tool Calling Workflow

### What we are building:
Explain and demonstrate how Gemini invokes tools and processes observations via `ToolMessage`.

### Why we need it:
Tool calling empowers the LLM to inspect external state deterministically rather than relying on static training weights.

In [17]:
# Section 19: Tool Calling Workflow
print("Tools bound to LLM in TravelLogisticsAgent:")
for t_name, t_obj in logistics_agent.tools_by_name.items():
    print(f" • Name: {t_name}")
    print(f"   Schema: {t_obj.args}")

Tools bound to LLM in TravelLogisticsAgent:
 • Name: search_logistics_information
   Schema: {'query': {'description': 'Search query describing origin, destination, transport type, route, or schedule.', 'title': 'Query', 'type': 'string'}}
 • Name: search_transport_routes
   Schema: {'origin': {'description': "Origin city or location (e.g. 'Colombo', 'Kandy')", 'title': 'Origin', 'type': 'string'}, 'destination': {'description': "Destination city or location (e.g. 'Kandy', 'Ella', 'Galle')", 'title': 'Destination', 'type': 'string'}}
 • Name: search_transport_schedule
   Schema: {'origin': {'description': 'Origin city or station', 'title': 'Origin', 'type': 'string'}, 'destination': {'description': 'Destination city or station', 'title': 'Destination', 'type': 'string'}, 'transport_type': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': "Optional filter: 'Train' or 'Bus'", 'title': 'Transport Type'}}
 • Name: check_transport_availability
   Schema: {'or

---
## Section 20: Bounded Execution Guardrail

### What we are building:
Enforce `MAX_STEPS = 5` in `_agent_node` and `_route_tools`.

### Why we need it:
Protects against runaway agentic loops and infinite tool calls, ensuring bounded execution safety .

In [18]:
# Section 20: Bounded Execution Guardrail
print(f"🛡️ Guardrail Configuration: MAX_STEPS = {MAX_STEPS}")
print("If step count exceeds MAX_STEPS, the agent loop halts immediately and routes to validation_node with safety notice.")

🛡️ Guardrail Configuration: MAX_STEPS = 5
If step count exceeds MAX_STEPS, the agent loop halts immediately and routes to validation_node with safety notice.


---
## Section 21: Error Handling & Safe Fallbacks

### What we are building:
Input validation guards for empty strings, missing locations, and tool failures.

### Why we need it:
Production agents must fail safely without unhandled exceptions.

In [19]:
# Section 21: Error Handling Test
empty_test_result = logistics_agent.run("")
print("Empty Query Fallback Output:")
print(json.dumps(empty_test_result["result"], indent=2))

Empty Query Fallback Output:
{
  "origin": "Unknown",
  "destination": "Unknown",
  "travel_date": null,
  "transport_options": [],
  "information_limitations": [
    "ERROR: Input request is empty. Please provide a valid transit inquiry."
  ],
  "unsupported_requests_note": "Input Validation Error: Empty Query"
}


---
## Section 22: Output Validation Node

### What we are building:
Validate the agent's final text into `LogisticsResult` Pydantic model.

### Why we need it:
Guarantees schema compliance and graceful fallback if the LLM produces malformed JSON.

In [20]:
# Section 22: Output Validation Node
print("Output validation node is active in the LangGraph graph, parsing and guaranteeing LogisticsResult output.")

Output validation node is active in the LangGraph graph, parsing and guaranteeing LogisticsResult output.


---
## Section 23: Comprehensive Test Cases

We evaluate the Travel Logistics & Availability Agent across all 10 required test scenarios:
- **TEST 1**: Colombo to Kandy (Transport options)
- **TEST 2**: Kandy to Ella (Train information)
- **TEST 3**: Colombo to Galle (Public buses)
- **TEST 4**: Kandy to Ella (Travel duration)
- **TEST 5**: Kandy to Ella tomorrow morning (Live data limitation guardrail)
- **TEST 6**: Create 5-day itinerary (Boundary: Travel Planning Agent)
- **TEST 7**: Which destination is best (Boundary: Recommendation Agent)
- **TEST 8**: Book me a train ticket (Boundary: Booking disclaimed)
- **TEST 9**: Empty request (Input validation error)
- **TEST 10**: Unavailable corridor (Safe missing data notice, no hallucination)

In [21]:
# Section 23: Execute Test Cases
test_queries = [
    ("TEST 1: Transport options", "How can I travel from Colombo to Kandy?"),
    ("TEST 2: Train information", "Show me train information from Kandy to Ella."),
    ("TEST 3: Public buses", "What public buses can I take from Colombo to Galle?"),
    ("TEST 4: Travel duration", "What is the estimated travel duration from Kandy to Ella?"),
    ("TEST 5: Live data guardrail", "Are there trains from Kandy to Ella tomorrow morning?"),
    ("TEST 6: Itinerary boundary", "Create a 5-day Sri Lanka itinerary."),
    ("TEST 7: Recommendation boundary", "Which destination is best for me?"),
    ("TEST 8: Booking boundary", "Book me a train ticket from Colombo to Kandy."),
    ("TEST 9: Empty request", ""),
    ("TEST 10: Unavailable corridor", "How can I travel from Jaffna to Hambantota?")
]

test_results = []
for label, q in test_queries:
    print(f"\n{'='*70}\n🚀 Running {label}: '{q}'\n{'='*70}")
    res = logistics_agent.run(q)
    output_obj = res.get("result", {})
    test_results.append((label, q, output_obj))
    print(f"Result Origin: {output_obj.get('origin')} -> Destination: {output_obj.get('destination')}")
    print(f"Transport Options Found: {len(output_obj.get('transport_options', []))}")
    if output_obj.get("unsupported_requests_note"):
        print(f"Boundary Note: {output_obj.get('unsupported_requests_note')}")
    if output_obj.get("information_limitations"):
        print(f"Limitations: {output_obj.get('information_limitations')}")


🚀 Running TEST 1: Transport options: 'How can I travel from Colombo to Kandy?'


c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Result Origin: Colombo -> Destination: Kandy
Transport Options Found: 2
Boundary Note: Ticket booking functionality is not currently connected to live booking engines. Live seat availability requires official live API integration.

🚀 Running TEST 2: Train information: 'Show me train information from Kandy to Ella.'


c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Result Origin: Kandy -> Destination: Ella
Transport Options Found: 1
Limitations: ['Live, real-time seat availability and exact real-time delays cannot be verified without official live API connections.']

🚀 Running TEST 3: Public buses: 'What public buses can I take from Colombo to Galle?'


...

Result Origin: Inferred from query -> Destination: Inferred from query
Transport Options Found: 0
Boundary Note: Destination recommendations are handled by the Recommendation & Feedback Analysis Agent.
Limitations: ['Response structured according to safety fallback schema.', 'Summary message: Personalized destination recommendations, review sentiment analysis, and suitability scoring are handled by the Recommendation & Feedback Analysis Agent. \n\nAs the Travel Logistics & Availability Agent, my role is to provide factual travel logistics, public transit routes, timetab']

🚀 Running TEST 8: Booking boundary: 'Book me a train ticket from Colombo to Kandy.'


c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Result Origin: Colombo -> Destination: Kandy
Transport Options Found: 1
Boundary Note: Ticket booking functionality is not currently connected to live booking engines. I can provide schedule and availability guidance, but cannot process reservations.

🚀 Running TEST 9: Empty request: ''
Result Origin: Unknown -> Destination: Unknown
Transport Options Found: 0
Boundary Note: Input Validation Error: Empty Query
Limitations: ['ERROR: Input request is empty. Please provide a valid transit inquiry.']

🚀 Running TEST 10: Unavailable corridor: 'How can I travel from Jaffna to Hambantota?'


c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Result Origin: Jaffna -> Destination: Hambantota
Transport Options Found: 0
Limitations: ['No direct transit routes, schedules, or direct transport options between Jaffna and Hambantota exist in the prototype knowledge base.', 'Live, real-time seat availability cannot be verified without official live API connections.']


---
## Section 24: Agent Evaluation

### What we are building:
Structured evaluation table assessing the agent against 8 core Agentic AI criteria.

In [22]:
# Section 24: Agent Evaluation
eval_criteria = [
    ("1. Tool Usage", "Pass", "Invokes search_logistics_information, search_routes, or search_schedules deterministically."),
    ("2. Retrieval Relevance", "Pass", "FAISS vector store returns relevant transit records."),
    ("3. Factual Grounding", "Pass", "All answers grounded in prototype knowledge base; no invented train numbers."),
    ("4. Structured Output Validity", "Pass", "All responses conform to LogisticsResult Pydantic schema."),
    ("5. Responsibility Boundaries", "Pass", "Refuses itinerary generation, recommendation scoring, and bookings."),
    ("6. Hallucination Avoidance", "Pass", "Correctly identifies unavailable corridors (e.g. Jaffna to Hambantota)."),
    ("7. Error Handling", "Pass", "Handles blank queries safely with schema fallback."),
    ("8. Bounded Execution", "Pass", "Enforces MAX_STEPS = 5 limit against runaway loops.")
]

print(f"{'CRITERION':<32} | {'STATUS':<6} | {'EVALUATION SUMMARY'}")
print("-" * 80)
for crit, stat, summary in eval_criteria:
    print(f"{crit:<32} | {stat:<6} | {summary}")

CRITERION                        | STATUS | EVALUATION SUMMARY
--------------------------------------------------------------------------------
1. Tool Usage                    | Pass   | Invokes search_logistics_information, search_routes, or search_schedules deterministically.
2. Retrieval Relevance           | Pass   | FAISS vector store returns relevant transit records.
3. Factual Grounding             | Pass   | All answers grounded in prototype knowledge base; no invented train numbers.
4. Structured Output Validity    | Pass   | All responses conform to LogisticsResult Pydantic schema.
5. Responsibility Boundaries     | Pass   | Refuses itinerary generation, recommendation scoring, and bookings.
6. Hallucination Avoidance       | Pass   | Correctly identifies unavailable corridors (e.g. Jaffna to Hambantota).
7. Error Handling                | Pass   | Handles blank queries safely with schema fallback.
8. Bounded Execution             | Pass   | Enforces MAX_STEPS = 5 limit agai

---
## Section 25: Safe Operational Execution Trace

### What we are building:
Demonstrate the operational event log for a complex transit query, highlighting steps taken without exposing private chain-of-thought.

In [23]:
# Section 25: Safe Execution Trace Demonstration
demo_tracer = SafeExecutionTracer(verbose=True)
demo_agent = TravelLogisticsAgent(
    model_name="gemini-3.5-flash-lite",
    temperature=0.0,
    retriever=retriever,
    tracer=demo_tracer
)

print("--- Running Trace Demonstration on 'Colombo to Kandy train options' ---")
trace_result = demo_agent.run("How can I travel from Colombo to Kandy by train?")
print(f"\nTotal Steps Logged: {len(demo_tracer.get_logs())}")

--- Running Trace Demonstration on 'Colombo to Kandy train options' ---


c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[AGENT TRACE | Step 1] Action: LLM Reasoning Step
   • has_tool_calls: True
   • tool_calls_requested: ['search_transport_routes']

[AGENT TRACE | Step 1] Action: Tool Executed Successfully
   • tool: search_transport_routes
   • args: {'origin': 'Colombo', 'destination': 'Kandy'}
   • observation_preview: Found 2 route option(s) connecting Colombo and Kandy:

Option 1: [Train] Main Line (Colombo Fort - Badulla / Kandy)
  • ...


c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[AGENT TRACE | Step 2] Action: LLM Reasoning Step
   • has_tool_calls: True
   • tool_calls_requested: ['search_transport_schedule']

[AGENT TRACE | Step 2] Action: Tool Executed Successfully
   • tool: search_transport_schedule
   • args: {'transport_type': 'Train', 'destination': 'Kandy', 'origin': 'Colombo'}
   • observation_preview: Found 1 prototype schedule entry/entries for Colombo to Kandy:

[1] Train - Intercity & Express Trains (Podi Menike, Uda...


c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[AGENT TRACE | Step 3] Action: LLM Reasoning Step
   • has_tool_calls: True
   • tool_calls_requested: ['check_transport_availability']

[AGENT TRACE | Step 3] Action: Tool Executed Successfully
   • tool: check_transport_availability
   • args: {'transport_type': 'Train', 'destination': 'Kandy', 'origin': 'Colombo'}
   • observation_preview: Availability status for Colombo to Kandy (Train):

[1] Train - Intercity & Express Trains (Podi Menike, Udarata Menike)
...


c:\Users\user\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[AGENT TRACE | Step 4] Action: LLM Reasoning Step
   • has_tool_calls: False
   • tool_calls_requested: []

[AGENT TRACE | Step 4] Action: Output Validation Passed
   • transport_options_count: 1

Total Steps Logged: 8


---
## Section 26: Final Structured Output Demonstration

### What we are building:
Display the final JSON payload produced for consumption by the Travel Planning Agent.

In [24]:
# Section 26: Final Structured Output Demonstration
final_payload = trace_result["result"]
print(json.dumps(final_payload, indent=2))
print("\n✅ Verification Complete: Travel Logistics & Availability Agent is fully operational!")

{
  "origin": "Colombo",
  "destination": "Kandy",
  "travel_date": null,
  "transport_options": [
    {
      "transport_type": "Train",
      "route": "Main Line (Colombo Fort - Kandy)",
      "departure_location": "Colombo Fort Railway Station",
      "arrival_location": "Kandy Railway Station",
      "departure_time": "05:55 AM, 07:00 AM, 08:30 AM, 10:35 AM, 03:35 PM, 05:45 PM",
      "arrival_time": "Varies by departure (approx. 2.5 to 3.25 hrs later)",
      "estimated_duration": "2 hours 30 minutes to 3 hours 15 minutes",
      "availability": "Scheduled daily operations in prototype knowledge base. Real-time seat availability cannot be verified without live railway API integration.",
      "source": "prototype knowledge base"
    }
  ],
  "information_limitations": [
    "Live, real-time seat availability and exact live train delay tracking are not available through the prototype knowledge base."
  ],
  "unsupported_requests_note": "Ticket booking functionality is not currently